# 🧠 AI Logic Engine: Knowledge Ingestion (Speed Optimized)

### 🛡️ Final Robustness Pass (v3.9):
- **Live Debug:** AI က ဘာတွေပြန်ဖြေနေလဲဆိုတာကို အောက်မှာ တန်းပြပါမယ်။ (AI စကားပြောမပြော အမြဲစစ်လို့ရပါတယ်)
- **Smart Prompt:** AI ကို Subject|Verb|Object format ကွဲမထွက်သွားအောင် ပိုတင်းကျပ်တဲ့ ညွှန်ကြားချက်ပေးထားပါတယ်။
- **Mobile Optimized:** Mobile browser တွေမှာ hang တတ်တဲ့ input code တွေကို ရှင်းထုတ်ထားပါတယ်။

In [ ]:
# @title 🛠️ Step 1: Install & Imports
!pip install -q firebase-admin transformers accelerate bitsandbytes torch tqdm

import os, json, re, torch, time
from tqdm.auto import tqdm
import firebase_admin
from firebase_admin import credentials, firestore
from google.colab import files
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("✅ Core Libraries Loaded.")

In [ ]:
# @title 🔑 Step 2: Firestore Connection
DATABASE_ID = 'ai-studio-09d97652-b1c3-4b1a-b63e-31d8a38be4c2'

if not firebase_admin._apps:
    print("Service Account Key (JSON) ကို Upload တင်ပေးပါ...")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("❌ JSON key မရှိဘဲ ဆက်သွား၍မရပါ။")
    filename = list(uploaded.keys())[0]
    cred = credentials.Certificate(filename)
    firebase_admin.initialize_app(cred)

db = firestore.client(database_id=DATABASE_ID)
print(f"✅ Memory Synced: {DATABASE_ID}")

In [ ]:
# @title 🧬 Step 3: Logic Engine Config
STATE_FILE = "ingestion_state_v3_9.json"

def clean_id(text):
    if not text: return ""
    text = str(text).lower().strip()
    return re.sub(r'[^a-z0-9]+', '_', text).strip('_')

def normalize_verb(v):
    v = str(v).strip().lower()
    is_a_set = ['is_a', 'is a', 'is type of', 'belongs to', 'category of']
    if any(p == v for p in is_a_set): return 'is_a'
    return v.replace(' ', '_')

def save_state(count):
    with open(STATE_FILE, "w") as f: json.dump({"count": count}, f)

def load_state():
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r") as f: return json.load(f).get("count", 0)
        except: pass
    return 0

print("✅ Standard Normalizer Configured.")

In [ ]:
# @title 🤖 Step 4: Load Symbolic Model
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"⏳ Loading {model_id}... ")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_compute_dtype=torch.float16, 
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None: 
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config, 
    device_map="auto"
)
model.eval()
print("✅ AI Logic Engine is fully active.")

In [ ]:
# @title 🚀 Step 5: Start Continuous Ingestion
TARGET_FACTS = 1000000 # @param {type:"number"}
CATEGORIES = ["General Knowledge", "Small Talk Greetings", "Social Manners", "Pop Culture trivia", "Daily Conversations", "Travel phrases"]

count = load_state()
pbar = tqdm(initial=count, total=TARGET_FACTS, desc="Total Symbols")

while count < TARGET_FACTS:
    try:
        cat = CATEGORIES[count % len(CATEGORIES)]
        # Strict prompt to force PIPE output
        prompt = f"Generate 20 factual triplets about {cat}. Format: Subject|Verb|Object|Sentence. No numbering, no chat, just raw pipes."
        messages = [{"role": "system", "content": "Output S|V|O|Sentence only. No intros."}, {"role": "user", "content": prompt}]
        
        it = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inp = tokenizer(it, return_tensors="pt", padding=True).to(model.device)
        
        with torch.no_grad():
            out = model.generate(inp.input_ids, attention_mask=inp.attention_mask, max_new_tokens=800, do_sample=True, temperature=0.7, pad_token_id=tokenizer.pad_token_id)
        
        response = tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
        
        # Debug: Show AI output
        print(f"\n🤖 AI Result for {cat}:")
        print(response.strip()[:200] + "...")
        
        batch = db.batch()
        added_this_loop = 0
        
        for line in response.strip().split('\n'):
            if '|' not in line: continue
            parts = [p.strip() for p in line.split('|')]
            if len(parts) >= 3:
                s, v, o = parts[0], normalize_verb(parts[1]), parts[2]
                sid, oid = clean_id(s), clean_id(o)
                if sid and oid and sid != oid:
                    node_ref = db.collection('nodes').document(sid)
                    data = {'name': s, 'updatedAt': firestore.SERVER_TIMESTAMP}
                    if v == 'is_a':
                        data['groups'] = firestore.ArrayUnion([oid])
                    else:
                        sent = parts[3] if len(parts) > 3 else f"{s} {v.replace('_',' ')} {o}."
                        data['relations'] = firestore.ArrayUnion([{'verb': v, 'targetId': oid, 'sentence': sent, 'weight': 100}])
                    batch.set(node_ref, data, merge=True)
                    added_this_loop += 1
        
        if added_this_loop > 0:
            batch.commit()
            count += added_this_loop
            pbar.update(added_this_loop)
            save_state(count)
            print(f"✅ Syncing {added_this_loop} facts...")
        else:
            print("⚠️ No usable data found in this chunk. Retrying...")

        if count % 20 == 0: torch.cuda.empty_cache()
        
    except Exception as e:
        print(f"\n❌ Error: {e}. Cooling down 3s...")
        time.sleep(3)
    except KeyboardInterrupt:
        break

pbar.close()
print(f"✅ Loop ended. Final count: {count}")